# Repo Market Model — an overview

This notebook runs from a fresh clone in well under a minute, on Colab or on your own machine. It rebuilds the
project's data panel from the files tracked in the repository, checks it against the published digest, and then walks
through what the project is about, what the data say, and where the work stands.

**This notebook is a pre-as-of illustration, pinned to panel 4ddc3882.** It reads the rebuilt panel directly, not
through the project's loader and its lockbox guard (`require_unlocked`), and its model figures are scouting
measurements that decide nothing. The published results are in the [project guide](../PROJECT_GUIDE.md) and `docs/runs/`, scored under the as-of
rule. Section 5 shows why the as-of rule matters.

## 1. What repo pressure is

Every day, banks, securities dealers and money-market funds lend each other roughly two trillion dollars overnight,
against U.S. Treasury securities as collateral. That is the **repo market** (short for *repurchase agreement*: sell a
Treasury today, buy it back tomorrow). The New York Fed publishes the average rate on that borrowing each morning as
**SOFR**, the Secured Overnight Financing Rate.

The Federal Reserve pays banks interest on the cash they keep at the Fed. That rate is **IORB**, interest on reserve
balances (before July 2021 it was called IOER). A bank with spare cash can leave it at the Fed and earn IORB, or lend it
in repo. So when cash is plentiful, SOFR sits at or a little below IORB.

When spare cash runs short, borrowers have to bid for it and SOFR rises above IORB. That usually happens on days when a
lot of cash leaves the system at once: corporate tax dates, large Treasury settlements, month-ends and quarter-ends. In
September 2019 all of them coincided with low reserves, and SOFR jumped to more than three percentage points above the
Fed's rate in a single day.

**This project calls a day a pressure day when SOFR − IORB, rounded to whole basis points, is strictly above +5** (a basis point is 0.01
percentage points; a day exactly on +5 is not one). Its headline question is: *from what is public at 4 pm today, what is the chance tomorrow is a
pressure day?*

It is an academic and portfolio project. Nothing here is investment advice or a risk-management tool.

## 2. Get the code and rebuild the data

The next cell finds the repository if you opened the notebook inside a clone, and clones it otherwise (the Colab
case). Then it runs the build command from `REPRODUCIBILITY.md`, which assembles the daily panel from the snapshot
files tracked under `tests/fixtures/`. No network access to any data provider is needed, and nothing is installed:
the package itself uses only the Python standard library. The last step checks the panel's SHA-256 digest against the
published manifest, so you know you are looking at exactly the data every record was scored on.

In [ ]:
import hashlib
import os
import pathlib
import subprocess
import sys
import tempfile
import time

REPO_URL = "https://github.com/eleonorabjornberg/repo-market-model.git"
EXPECTED_DIGEST = "4ddc3882"  # the published panel, metadata/funding_panel_manifest.json

started = time.time()


def find_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "repo_model").is_dir() and (candidate / "metadata").is_dir():
            return candidate
    return None


root = find_root(pathlib.Path.cwd().resolve())
if root is None:
    root = pathlib.Path.cwd().resolve() / "repo-market-model"
    if not root.exists():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO_URL, str(root)], check=True)
os.chdir(root)
sys.path.insert(0, str(root / "src"))
print("repository:", root)

In [ ]:
PANEL = pathlib.Path(tempfile.gettempdir()) / "funding_panel.csv"
env = dict(os.environ, PYTHONPATH=str(root / "src"))


def cli(*args):
    done = subprocess.run([sys.executable, "-m", "repo_model.cli", *args],
                          cwd=root, env=env, capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(done.stderr)
    return done.stdout


cli("build", "--raw-root", "tests/fixtures/snapshots/funding_inputs", "--output", str(PANEL),
    "--build-cutoff", "2026-09-08T21:31:42+00:00", "--decision-time", "16:00:00")
cli("verify-panel", str(PANEL), "--manifest", "metadata/funding_panel_manifest.json")

digest = hashlib.sha256(PANEL.read_bytes()).hexdigest()
assert digest.startswith(EXPECTED_DIGEST), digest
print(f"panel verified: sha256 {digest[:12]}…, built in {time.time() - started:.0f} s")

## 3. The data: what, why and how

The data selection is **Nicholas Beroud**'s, the project's advisor: which public series carry the funding market's
mechanics, and what each one means. The modelling, methodology and evaluation are **Eleonora Björnberg**'s. The
project guide's *People* section has the full split.

Every input is an official public source, saved as an immutable, checksummed snapshot, and each value carries the
moment it first became public. The inputs fall into four groups, each with a one-line reason for being there.

In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "docs" / "figures" / "overview-light.svg")))

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 80)
sources = pd.DataFrame(
    [
        ("Price of overnight cash", "SOFR, its 25th/75th percentiles and volume; TGCR; BGCR",
         "New York Fed, about 08:00 the next business day, final by 15:00",
         "The target itself, and its closest relatives"),
        ("Price of overnight cash", "IORB (IOER before July 2021)", "FRED, administered rate",
         "The rate the target is measured against"),
        ("Spare cash in the system", "Reserve balances, Treasury General Account (TGA)",
         "Fed H.4.1 through FRED, weekly",
         "Less spare cash means borrowers bid harder for what is left"),
        ("Spare cash in the system", "ON RRP take-up", "FRED (daily); not yet in the panel",
         "Cash parked at the Fed's facility is cash that could come back to repo"),
        ("Cash needed at once", "Month-end, quarter-end, corporate tax dates", "The calendar, known in advance",
         "Balance-sheet and tax days drain cash on known dates"),
        ("Cash needed at once", "Treasury settlements: bills, coupons, SOMA", "Treasury auction results",
         "Buyers of new Treasuries pay cash on settlement day"),
        ("Where cash can park instead", "4-week and 13-week Treasury bill yields",
         "Treasury daily bill rates, 16:30 on the quote date",
         "When bills pay more than repo, money funds move cash out of repo"),
        ("Also in the panel", "Primary dealers' Treasury positions", "New York Fed FR 2004, weekly",
         "Dealers finance their inventory in repo"),
    ],
    columns=["group", "series", "source and publication", "why it is here"],
)
sources

In [ ]:
panel = pd.read_csv(PANEL, parse_dates=["date"])
panel["spread_bps"] = (panel.sofr - panel.iorb) * 100
print(f"{len(panel)} business days, {panel.date.min():%Y-%m-%d} to {panel.date.max():%Y-%m-%d}")
panel.tail(3)

## 4. The data by year: pressure days and the reserves regime

Pressure is not spread evenly through time. It clustered in 2018–19, when reserves were falling towards what turned out
to be scarcity, vanished after the Fed flooded the system with reserves in 2020, and came back in 2025. A *scheduled*
pressure day here is a month-end (the last two business days), a quarter-end or a corporate tax date: a day on which
cash is known in advance to be in demand.

In [ ]:
import matplotlib.pyplot as plt

panel["year"] = panel.date.dt.year
from repo_model.data import in_last_business_days_of_month  # the last two business days, on the market calendar
panel["scheduled"] = (panel.date.dt.date.map(in_last_business_days_of_month)) | (panel.quarter_end == 1) | (panel.tax_date == 1)
by_year = panel.groupby("year").agg(
    days=("date", "size"),
    sofr_above_iorb=("spread_bps", lambda s: int((s > 0).sum())),
    pressure_days=("spread_bps", lambda s: int((s.round() > 5).sum())),
    reserves_median_tn=("reserve_balances", lambda s: round(s.median() / 1e3, 2)),
)
by_year["pressure_on_scheduled_days"] = panel[panel.spread_bps.round() > 5].groupby("year").scheduled.sum()
by_year = by_year.fillna(0).astype({"pressure_on_scheduled_days": int})
by_year

In [ ]:
fig, (top, bottom) = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True,
                                  gridspec_kw={"height_ratios": [3, 2]})
top.plot(panel.date, panel.spread_bps.clip(upper=40), lw=0.7, color="#2a78d6")
top.axhline(5, color="#d6542a", lw=1, ls="--", label="pressure: +5 bp")
top.axhline(0, color="#898781", lw=0.6)
top.set_ylabel("SOFR − IORB, bp\n(clipped at 40)")
top.legend(loc="upper right", frameon=False)
bottom.plot(panel.date, panel.reserve_balances / 1e3, color="#52514e", lw=1)
bottom.set_ylabel("reserves, $ tn")
for ax in (top, bottom):
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Pressure days and reserves", x=0.01, ha="left")
fig.tight_layout()

## 5. What is known at 16:00 the day before

The project forecasts each business day **T** from what is public at **16:00 on the business day before**. Each input has
its own publication time, so "what is known" differs by input:

- **Rates** (SOFR, TGCR, BGCR): the latest value published and final by 16:00 on T−1 is the one *for* T−2. SOFR for a day
  comes out at about 08:00 the next business day and is final by 15:00.
- **Bill yields** are published at 16:30 on their quote date, so at 16:00 on T−1 the latest is T−2's too.
- **Weekly balance-sheet series** (reserves, TGA): the latest weekly print released by then.
- **Scheduled inputs** refer to T itself and are known in advance: the calendar always, Treasury settlements once the
  auction results are out.

That is the **as-of information rule** (`docs/decisions/information-set.md`), and the published records are scored under it.

**The published records did not read inputs this way.** They turned SOFR's one-*business*-day lag into its worst case in
*calendar* days (six) and used that number to choose the feature row too, so every published forecast read the row
dated at least seven calendar days before T. The old design never leaked, but it was about a week stale. The cell below
shows the row each rule reads for one day of the September 2019 episode, and how many rows back each rule reaches
across the sample.

In [ ]:
import numpy as np

dates = panel.date.values.astype("datetime64[D]")
n = len(panel)
rows = np.arange(n)
published_row = np.searchsorted(dates, dates - np.timedelta64(6, "D"), side="left") - 1  # dated <= T - 7 days
asof_row = rows - 2  # the value for T-2, final at 15:00 on T-1

T = int(np.where(panel.date == pd.Timestamp("2019-09-17"))[0][0])
example = pd.DataFrame({
    "rule": ["as-of rule", "published records (pre-as-of)"],
    "reads SOFR for": [panel.date[asof_row[T]].date(), panel.date[published_row[T]].date()],
    "rows before T": [T - asof_row[T], T - published_row[T]],
})
print(f"Scored day T = {panel.date[T]:%a %d %b %Y}; decision at 16:00 on {panel.date[T - 1]:%a %d %b %Y}")
display(example)

first = int(np.where(panel.date == pd.Timestamp("2018-07-05"))[0][0])
scored = np.arange(first, n)
gap = pd.Series(scored - published_row[scored]).value_counts().sort_index()
gap.index.name = "rows between the published rule's feature row and T"
gap.rename("scored days").to_frame()

## 6. The lag finding, in one cell

*A scouting figure from before the as-of rule was implemented, not a record.*

Persistence ("tomorrow looks like the latest value you have") is the benchmark every model has to beat. Scored on the
same 2,039 days from 2018-07-05, the only difference between the two lines below is which row it reads. The source is
`docs/pivot/scouting/`, and `docs/pivot/lag-assessment.md` has the full account.

In [ ]:
y = panel.spread_bps.values
lag = pd.DataFrame({
    "persistence reads": ["published rule (≥ 7 calendar days back)", "as-of rule (T−2)"],
    "MAE, bp": [np.mean(np.abs(y[scored] - y[published_row[scored]])),
                np.mean(np.abs(y[scored] - y[asof_row[scored]]))],
})
assert len(scored) == 2039
assert abs(lag["MAE, bp"][0] - 3.104) < 0.0005 and abs(lag["MAE, bp"][1] - 2.509) < 0.0005
print(f"{len(scored)} scored days, {panel.date[first]:%Y-%m-%d} to {panel.date[n - 1]:%Y-%m-%d}  "
      "(pre-as-of scouting figure)")
lag.round(3)

Reading the same number at the right time cuts the benchmark's average error by about a fifth. Every published
comparison set a handicapped model against a handicapped benchmark, which is why the published model figures were re-scored under the as-of rule rather than carried over.

## 7. Pressure-probability baselines, scored in seconds

*Pre-as-of scouting figures, not records.*

The headline target is a probability: P(SOFR − IORB > +5 bp on T, on whole basis points), and the same at +10 bp. Before any model is allowed to
claim skill it has to beat two baselines:

- **Climatology**: the share of past days that were pressure days, over everything observable at the decision.
- **Persistence-logistic**: a one-variable logistic regression on the latest spread, refitted every 21 scored days on
  labels that were already public at the refit.

Scores: the **Brier score** (mean squared error of the probability; lower is better), **skill against climatology**
(1 − Brier / climatology's Brier; above zero beats it) and **AUROC** (how well the probability ranks pressure days above
the rest; 0.5 is a coin toss).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

BLOCK = 21  # refit every 21 scored days


def persistence_logistic(x, threshold):
    p = np.full(n, np.nan)
    for start in range(0, len(scored), BLOCK):
        T0 = scored[start]
        block = scored[start:start + BLOCK]
        train = np.arange(2, T0 - 1)  # labels final by the block's decision instant
        keep = ~np.isnan(x[train])
        labels = (np.round(y[train][keep]) > threshold).astype(int)
        if labels.min() == labels.max():  # no event yet: the only honest probability is the constant
            p[block] = labels[0]
            continue
        model = LogisticRegression().fit(x[train][keep].reshape(-1, 1), labels)
        p[block] = model.predict_proba(x[block].reshape(-1, 1))[:, 1]
    return p


x_asof = np.r_[np.nan, np.nan, y[:-2]]
x_published = np.where(published_row >= 0, y[np.clip(published_row, 0, n - 1)], np.nan)

results = []
for threshold in (5, 10):
    event = (np.round(y) > threshold).astype(float)
    climatology = np.array([event[2:t - 1].mean() if t > 3 else np.nan for t in range(n)])
    reference = np.mean((climatology[scored] - event[scored]) ** 2)
    for name, p in [("climatology", climatology),
                    ("persistence-logistic, published rule", persistence_logistic(x_published, threshold)),
                    ("persistence-logistic, as-of", persistence_logistic(x_asof, threshold))]:
        brier = np.mean((p[scored] - event[scored]) ** 2)
        results.append({"threshold": f"> {threshold} bp", "events": int(event[scored].sum()), "model": name,
                        "Brier": brier, "skill vs climatology": 1 - brier / reference,
                        "AUROC": roc_auc_score(event[scored], p[scored])})

print(f"{len(scored)} scored days (pre-as-of scouting figure)")
pd.DataFrame(results).round({"Brier": 4, "skill vs climatology": 3, "AUROC": 3})

The one-variable logistic on the latest spread is already a strong baseline, and it is much stronger when it reads the
spread at the right time. Any richer pressure model has to beat the as-of line, on reliability as well as on ranking,
and split by regime and by type of day, before it can claim anything.

## 8. Where the work stands

The README's generated status block says which phase is current, and `docs/status.json` carries the same in machine-readable
form. The published records are in `docs/runs/`, the plan is `PLAN.md`, and the decisions that govern the work are in
`docs/decisions/`. This notebook keeps no roadmap of its own, because a copy would go stale.

In [ ]:
print(f"notebook finished in {time.time() - started:.0f} s")